In [1]:
!pip uninstall -y torchvision torchaudio
!pip install transformers accelerate safetensors rich

Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128


In [2]:
import collections
import gc
from typing import List, Optional
import torch
import torch.nn as nn
import torch.nn.functional as F

class LRUCache:
    """Tracks active experts and dictates eviction policy based on temporal locality."""
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache: collections.OrderedDict[int, int] = collections.OrderedDict()
        self.free_slots = list(range(capacity))

    def get_slot(self, expert_id: int) -> tuple[int, bool, Optional[int]]:
        if expert_id in self.cache:
            self.cache.move_to_end(expert_id)
            return self.cache[expert_id], True, None

        if self.free_slots:
            slot_idx = self.free_slots.pop(0)
            self.cache[expert_id] = slot_idx
            return slot_idx, False, None

        victim_expert_id, slot_idx = self.cache.popitem(last=False)
        self.cache[expert_id] = slot_idx
        return slot_idx, False, victim_expert_id

class BufferManager:
    """Manages the physical transition of weights between Host RAM and GPU VRAM."""
    def __init__(self, num_slots, num_experts, output_size, input_size, dtype, device, weights_src, layer_id):
        self.num_slots = num_slots
        self.device = device
        self.dtype = dtype
        self.num_experts = num_experts
        self.layer_id = layer_id

        self.lru = LRUCache(num_slots)
        self.buffers = torch.empty((num_slots, output_size, input_size), dtype=dtype, device=device)
        self.pageable_weights = weights_src.detach().cpu().clone()

    def load_expert(self, expert_id: int, presentation_mode: bool = False) -> tuple[int, bool]:
        slot_idx, is_hit, victim_id = self.lru.get_slot(expert_id)

        if not is_hit:
            if presentation_mode:
                if victim_id is not None:
                    print(f"   🧹 Cache Full! Evicting Expert {victim_id} from VRAM to Host.")
                print(f"   💾 Fetching Expert {expert_id} from Host RAM...")

            # Synchronous transfer for prototype safety
            self.buffers[slot_idx].copy_(self.pageable_weights[expert_id], non_blocking=False)
        else:
            if presentation_mode:
                print(f"   🟢 Cache HIT: Expert {expert_id} is already resident in VRAM.")

        return slot_idx, is_hit

class GraniteMoeParallelExpertsOptimized(nn.Module):
    """Overrides the standard MoE block to intercept routing decisions."""
    def __init__(self, num_experts, input_size, output_size, num_gpu_slots=3, layer_id=""):
        super().__init__()
        self.num_experts = num_experts
        self.input_size = input_size
        self.output_size = output_size
        self.layer_id = layer_id

        self.weight = nn.Parameter(torch.empty(num_experts, output_size, input_size))
        self.buffer_manager = None
        self.num_slots = min(max(1, num_gpu_slots), max(1, num_experts))
        self.presentation_mode = False

    def _init_buffer_manager(self, device: torch.device):
        if self.buffer_manager is None:
            self.buffer_manager = BufferManager(
                self.num_slots, self.num_experts, self.output_size, self.input_size,
                self.weight.dtype, device, self.weight.data, self.layer_id
            )
            # Free device memory instantly
            self.weight.data = torch.empty(0, device=device, dtype=self.weight.dtype)

    def forward(self, inputs: torch.Tensor, expert_size: List[int]) -> torch.Tensor:
        self._init_buffer_manager(inputs.device)
        input_list = inputs.split(expert_size, dim=0)
        out_chunks = [None] * self.num_experts
        required_experts = [i for i, size in enumerate(expert_size) if size > 0]

        for i in range(self.num_experts):
            if input_list[i].numel() == 0:
                out_chunks[i] = input_list[i].new_empty((0, self.output_size))

        resident_experts = [eid for eid in required_experts if eid in self.buffer_manager.lru.cache]
        cold_experts = [eid for eid in required_experts if eid not in self.buffer_manager.lru.cache]
        execution_order = resident_experts + cold_experts

        should_log = self.presentation_mode and self.layer_id == "model.layers.0.block_sparse_moe.experts"
        if should_log and required_experts:
            print(f"\n[Token Routing] Layer 0 requested Experts: {required_experts}")

        for expert_id in execution_order:
            slot_idx, _ = self.buffer_manager.load_expert(expert_id, presentation_mode=should_log)
            out_chunks[expert_id] = F.linear(input_list[expert_id], self.buffer_manager.buffers[slot_idx])

        return torch.cat(out_chunks, dim=0)

def apply_cacheflow(model: nn.Module, num_gpu_slots: int = 3, presentation_mode: bool = False):
    """Walks the model tree and injects the CacheFlow prototype."""
    replaced = 0
    def _replace(module, prefix=""):
        nonlocal replaced
        for name, child in list(module.named_children()):
            full_name = f"{prefix}.{name}" if prefix else name
            if child.__class__.__name__ == "GraniteMoeParallelExperts":
                opt_child = GraniteMoeParallelExpertsOptimized(
                    child.num_experts, child.input_size, child.output_size, num_gpu_slots, full_name
                )
                opt_child.load_state_dict(child.state_dict(), strict=True)
                opt_child.to(device=child.weight.device, dtype=child.weight.dtype)
                setattr(module, name, opt_child)
                del child
                replaced += 1
            else:
                _replace(child, full_name)

    _replace(model)

    for _, module in model.named_modules():
        if isinstance(module, GraniteMoeParallelExpertsOptimized):
            module.presentation_mode = presentation_mode
            module._init_buffer_manager(module.weight.device)

    if torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    print(f"✅ CacheFlow Hook Injected: {replaced} MoE Layers Optimized.")
    return model

In [3]:
import time
import rich
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
model_path = "ibm-granite/granite-3.1-3b-a800m-instruct"

print("📥 Loading Standard Granite-3B MoE...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, device_map=device)

if torch.cuda.is_available():
    baseline_vram = torch.cuda.max_memory_allocated() / (1024 ** 2)
    print("\n" + "="*50)
    print(f"🔴 BASELINE PEAK VRAM: {baseline_vram:.2f} MB")
    print("="*50)

📥 Loading Standard Granite-3B MoE...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


🔴 BASELINE PEAK VRAM: 6292.00 MB


In [4]:
print("🚀 Initializing CacheFlow Dynamic LRU Engine...")

# Apply our custom class to limit it to 3 experts per layer
model = apply_cacheflow(model, num_gpu_slots=3, presentation_mode=True)

if torch.cuda.is_available():
    optimized_vram = torch.cuda.memory_allocated() / (1024 ** 2)
    print("\n" + "="*50)
    print(f"🟢 CACHEFLOW ACTIVE VRAM: {optimized_vram:.2f} MB")
    print("="*50)

🚀 Initializing CacheFlow Dynamic LRU Engine...
✅ CacheFlow Hook Injected: 64 MoE Layers Optimized.

🟢 CACHEFLOW ACTIVE VRAM: 963.95 MB


In [5]:
input_text = "Briefly explain the advantages of Mixture-of-Experts architecture."
input_tokens = {k: v.to(device) for k, v in tokenizer(input_text, return_tensors="pt").items()}

print("🧠 Commencing Token Generation...\n")

torch.cuda.reset_peak_memory_stats()
start_time = time.time()

output = model.generate(**input_tokens, max_new_tokens=60)

end_time = time.time()
speed = output.shape[1] / (end_time - start_time)

print("\n" + "="*50)
print(f"⚡ INFERENCE COMPLETE: {speed:.2f} Tokens/Sec")
print("="*50 + "\n")

rich.print(tokenizer.batch_decode(output)[0])

🧠 Commencing Token Generation...


⚡ INFERENCE COMPLETE: 2.97 Tokens/Sec



Briefly explain the advantages of Mixture-of-Experts architecture.

The Mixture-of-Experts (MoE) architecture is a powerful machine learning technique that offers several key 
advantages:

1. **Scalability**: MoE models can handle large-scale datasets more efficiently than traditional models. By 
distributing the computational load across multiple ex